In [2]:
import os
import glob
import pandas as pd
import numpy as np

def mode_series(s: pd.Series):
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]

def mode_share_series(s: pd.Series):
    vc = s.value_counts(dropna=False)
    return (vc.iloc[0] / vc.sum()) if len(vc) else 0.0

In [3]:
TRAIN_DIR = r"train_data"

### Находим все parquet-файлы train по маске и сортируем.

In [3]:
train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, "train_data_*.pq")))
print("files:", len(train_files))
print(*train_files[:3], sep="\n")

files: 12
train_data\train_data_0.pq
train_data\train_data_1.pq
train_data\train_data_10.pq


### Просмотр структуры

In [4]:
df0 = pd.read_parquet(train_files[0]) 
print(df0.shape)
display(df0.head(3))
print(df0.columns.tolist())
print(df0.dtypes.head(20))

(1974724, 61)


,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,...,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,0,1,18,9,2,3,16,10,11,3,...,3,3,3,4,1,3,4,1,0,0
1,0,2,18,9,14,14,12,12,0,3,...,0,0,0,4,1,3,4,1,0,0
2,0,3,18,9,4,8,1,11,11,0,...,0,0,0,4,1,2,3,1,1,1


['id', 'rn', 'pre_since_opened', 'pre_since_confirmed', 'pre_pterm', 'pre_fterm', 'pre_till_pclose', 'pre_till_fclose', 'pre_loans_credit_limit', 'pre_loans_next_pay_summ', 'pre_loans_outstanding', 'pre_loans_total_overdue', 'pre_loans_max_overdue_sum', 'pre_loans_credit_cost_rate', 'pre_loans5', 'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90', 'is_zero_loans5', 'is_zero_loans530', 'is_zero_loans3060', 'is_zero_loans6090', 'is_zero_loans90', 'pre_util', 'pre_over2limit', 'pre_maxover2limit', 'is_zero_util', 'is_zero_over2limit', 'is_zero_maxover2limit', 'enc_paym_0', 'enc_paym_1', 'enc_paym_2', 'enc_paym_3', 'enc_paym_4', 'enc_paym_5', 'enc_paym_6', 'enc_paym_7', 'enc_paym_8', 'enc_paym_9', 'enc_paym_10', 'enc_paym_11', 'enc_paym_12', 'enc_paym_13', 'enc_paym_14', 'enc_paym_15', 'enc_paym_16', 'enc_paym_17', 'enc_paym_18', 'enc_paym_19', 'enc_paym_20', 'enc_paym_21', 'enc_paym_22', 'enc_paym_23', 'enc_paym_24', 'enc_loans_account_holder_type', 'enc_loans_credit_status',

## Проверяем, пересекаются ли id между parquet-файлами. 
Это важно, потому что если один и тот же id встречается в нескольких файлах, простая агрегация по файлам (например, max или first) может дать некорректный результат. Функция проходит по каждому файлу, хранит уже встреченные id, и проверяет наличие пересечений.


In [5]:
def check_id_overlap(files, max_ids_store=2_000_000):
    seen = set()
    overlap_files = 0
    total_checked = 0
    
    for fp in files:
        ids = pd.read_parquet(fp, columns=["id"])["id"].dropna().unique()
        total_checked += len(ids)

        
        if len(seen) < max_ids_store:
            inter = np.intersect1d(ids, np.fromiter(seen, dtype=ids.dtype), assume_unique=False)
            if len(inter) > 0:
                overlap_files += 1
            seen.update(ids.tolist())
        else:
            sample = ids[:min(50_000, len(ids))]
            inter = np.intersect1d(sample, np.fromiter(seen, dtype=sample.dtype), assume_unique=False)
            if len(inter) > 0:
                overlap_files += 1
    
    print(f"Files checked: {len(files)}")
    print(f"Total unique-ids checked (sum over files, not global): {total_checked:,}")
    print(f"Files with overlap evidence: {overlap_files}")
    return overlap_files

overlap_files = check_id_overlap(train_files)

Files checked: 12
Total unique-ids checked (sum over files, not global): 3,000,000
Files with overlap evidence: 0


overlap_files == 0: Каждый id находится строго в одном файле.
Это позволяет безопасно агрегировать по файлам и затем объединять результаты.

## Проверяем корректность rn и “last”

In [6]:
def sanity_check_rn(files, n_files=3, n_rows=200_000):
    fps = files[:n_files]
    df = pd.concat([pd.read_parquet(fp, columns=["id","rn"]).sample(frac=1, random_state=0).head(n_rows) for fp in fps],
                   ignore_index=True)
    print("rn describe:\n", df["rn"].describe())
    dup = df.duplicated(["id","rn"]).mean()
    print(f"Duplicate (id,rn) share in sample: {dup:.4f}")

sanity_check_rn(train_files)

rn describe:
 count    600000.000000
mean          6.891423
std           5.323896
min           1.000000
25%           3.000000
50%           6.000000
75%          10.000000
max          44.000000
Name: rn, dtype: float64
Duplicate (id,rn) share in sample: 0.0000


### Анализ распределения rn показал:
- медиана ~6 наблюдений на клиента,
- 75% клиентов имеют ≤ 10 наблюдений,
- максимальное значение rn = 44.

Это подтверждает наличие истории по клиентам,
что делает агрегацию (mean, std, min, max, last) осмысленной.

Доля дубликатов (id, rn) = 0,
следовательно, определение "последнего состояния" через idxmax(rn) корректно.

## Сколько строк на один id 

In [7]:
def sanity_check_rows_per_id(files, n_files=3):
    fps = files[:n_files]
    parts = []
    for fp in fps:
        tmp = pd.read_parquet(fp, columns=["id"])
        parts.append(tmp["id"].value_counts())
    vc = pd.concat(parts).groupby(level=0).sum()
    print("Rows per id (sample over files) describe:\n", vc.describe(percentiles=[.5,.9,.95,.99]))

sanity_check_rows_per_id(train_files)

Rows per id (sample over files) describe:
 count    750000.000000
mean          8.504535
std           6.061046
min           1.000000
50%           7.000000
90%          17.000000
95%          20.000000
99%          26.000000
max          51.000000
Name: count, dtype: float64


## Анализ количества записей на одного клиента показал:
- медиана = 7 наблюдений,
- 90% клиентов имеют ≤ 17 наблюдений,
- максимальное количество наблюдений = 51.

Это подтверждает наличие исторической информации по клиентам.
Следовательно, агрегирование (mean, std, min, max, last)
является обоснованным и позволяет описать поведение клиента во времени.

In [5]:
df_raw = pd.read_parquet(train_files[0])
df_raw.dtypes


id                         int64
rn                         int64
pre_since_opened           int64
pre_since_confirmed        int64
pre_pterm                  int64
                           ...  
enc_loans_credit_status    int64
enc_loans_credit_type      int64
enc_loans_account_cur      int64
pclose_flag                int64
fclose_flag                int64
Length: 61, dtype: object

### Все признаки числовые, но есть категориальные уже закодированные - начинаются с enc, Для них агрегация по min max mean не осмыслено

In [6]:
[col for col in df_raw.columns if col.startswith("enc_")]

['enc_paym_0',
 'enc_paym_1',
 'enc_paym_2',
 'enc_paym_3',
 'enc_paym_4',
 'enc_paym_5',
 'enc_paym_6',
 'enc_paym_7',
 'enc_paym_8',
 'enc_paym_9',
 'enc_paym_10',
 'enc_paym_11',
 'enc_paym_12',
 'enc_paym_13',
 'enc_paym_14',
 'enc_paym_15',
 'enc_paym_16',
 'enc_paym_17',
 'enc_paym_18',
 'enc_paym_19',
 'enc_paym_20',
 'enc_paym_21',
 'enc_paym_22',
 'enc_paym_23',
 'enc_paym_24',
 'enc_loans_account_holder_type',
 'enc_loans_credit_status',
 'enc_loans_credit_type',
 'enc_loans_account_cur']

### Оценка памяти

In [8]:
mem_mb = df0.memory_usage(deep=True).sum() / 1024**2
print(f"df0 RAM ~ {mem_mb:.1f} MB")

df0 RAM ~ 919.0 MB


In [9]:
paym_cols = [f"enc_paym_{i}" for i in range(25)]
loan_cat_cols = [
    "enc_loans_account_holder_type",
    "enc_loans_credit_status",
    "enc_loans_credit_type",
    "enc_loans_account_cur",
]

BATCH_SIZE = 2 
parts = []
last_parts = []

for i in range(0, len(train_files), BATCH_SIZE):
    batch_files = train_files[i:i + BATCH_SIZE]

    if len(batch_files) == 1:
        df = pd.read_parquet(batch_files[0])
    else:
        df = pd.concat([pd.read_parquet(f) for f in batch_files], ignore_index=True)

    df["id"] = df["id"].astype("int32")
    df["rn"] = df["rn"].astype("int16")

    enc_cols = [c for c in df.columns if c.startswith("enc_")]
    
    g = df.groupby("id", sort=False)

    features = g.agg(
        loans_cnt=("rn", "size"),  
        rn_max=("rn", "max"),     
        rn_min=("rn", "min"),
    ).reset_index()

    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["id", "rn"] and c not in enc_cols]

    # считаем только sum/count/min/max (их можно корректно склеивать между батчами)
    if numeric_cols:
        num = g[numeric_cols].agg(["sum", "count", "min", "max"])
        num.columns = [f"{col}_{stat}" for col, stat in num.columns]
        num = num.reset_index()
        features = features.merge(num, on="id", how="left")

    # loan_cat_cols: считаем только count 
    present_loan_cat = [c for c in loan_cat_cols if c in df.columns]
    if present_loan_cat:
        loan_aggs = {f"{col}_cnt": (col, "count") for col in present_loan_cat}
        loan_df = g.agg(**loan_aggs).reset_index()
        features = features.merge(loan_df, on="id", how="left")

    # платежные признаки: obs/bad и их суммы по id 
    present_paym = [c for c in paym_cols if c in df.columns]
    if present_paym:
        X = df[present_paym]

        # obs_row = сколько из 25 статусов наблюдаемо в этой строке
        # bad_row = сколько наблюдаемых != 0 (трактуем как "плохие")
        obs_row = X.notna().sum(axis=1).astype("int32")
        bad_row = (X.notna() & X.ne(0)).sum(axis=1).astype("int32")

        tmp = pd.DataFrame({"id": df["id"].values,
                            "paym_obs": obs_row.values,
                            "paym_bad": bad_row.values})
        paym_df = tmp.groupby("id", sort=False).sum().reset_index()

        # share внутри батча НЕ финализируем, финализируем после глобального суммирования
        features = features.merge(paym_df, on="id", how="left")

    # last внутри батча (по максимальному rn) 
    idx = g["rn"].idxmax()
    last_rows = df.loc[idx, ["id", "rn"] + enc_cols].copy()
    last_rows = last_rows.rename(columns={c: f"last__{c}" for c in enc_cols})
    last_parts.append(last_rows)

    parts.append(features)
    print("batch", i // BATCH_SIZE, "->", features.shape)

# склеиваем батчи 
all_features = pd.concat(parts, ignore_index=True)

#  финальная агрегация по id 
agg = {"loans_cnt": "sum", "rn_max": "max", "rn_min": "min"}

for c in all_features.columns:
    if c in ("id", "loans_cnt", "rn_max", "rn_min"):
        continue

    if c.endswith(("_sum", "_count", "_cnt")):
        agg[c] = "sum"
    elif c.endswith("_min"):
        agg[c] = "min"
    elif c.endswith("_max"):
        agg[c] = "max"
    elif c in ("paym_obs", "paym_bad"):
        agg[c] = "sum"
    else:
        pass

final_features = all_features.groupby("id", as_index=False).agg(agg)

# считаем mean для numeric из sum/count 
sum_cols = [c for c in final_features.columns if c.endswith("_sum")]
for s in sum_cols:
    base = s[:-4]
    cnt = base + "_count"
    if cnt in final_features.columns:
        final_features[base + "_mean"] = final_features[s] / final_features[cnt].replace(0, np.nan)

# финализируем paym_bad_share после глобальных сумм 
if "paym_obs" in final_features.columns and "paym_bad" in final_features.columns:
    final_features["paym_obs_is0"] = (final_features["paym_obs"].fillna(0) == 0).astype("int8")
    final_features["paym_bad_share"] = (
        final_features["paym_bad"] / final_features["paym_obs"].replace(0, np.nan)
    ).fillna(0.0).astype("float32")

# глобальный last по всем батчам 
all_last = pd.concat(last_parts, ignore_index=True)
idx_global = all_last.groupby("id")["rn"].idxmax()
global_last = all_last.loc[idx_global].drop(columns=["rn"]).copy()

# финальный датасет
dataset = final_features.merge(global_last, on="id", how="left")
print("dataset:", dataset.shape)
dataset.head()

batch 0 -> (500000, 130)
batch 1 -> (500000, 130)
batch 2 -> (500000, 130)
batch 3 -> (500000, 130)
batch 4 -> (500000, 130)
batch 5 -> (500000, 130)
dataset: (3000000, 191)


,id,loans_cnt,rn_max,rn_min,pre_since_opened_sum,pre_since_opened_count,pre_since_opened_min,pre_since_opened_max,pre_since_confirmed_sum,pre_since_confirmed_count,...,last__enc_paym_19,last__enc_paym_20,last__enc_paym_21,last__enc_paym_22,last__enc_paym_23,last__enc_paym_24,last__enc_loans_account_holder_type,last__enc_loans_credit_status,last__enc_loans_credit_type,last__enc_loans_account_cur
0,0,10,10,1,81,10,1,18,76,10,...,3,4,3,3,3,4,1,2,4,1
1,1,14,14,1,160,14,2,18,107,14,...,3,4,3,3,3,4,1,2,3,1
2,2,3,3,1,25,3,0,13,32,3,...,3,4,3,3,3,4,1,2,3,1
3,3,15,15,1,105,15,1,18,110,15,...,3,4,3,3,3,4,1,2,4,1
4,4,1,1,1,12,1,12,12,9,1,...,3,4,3,3,3,4,1,2,3,1


## Агрегация событийных данных до уровня клиента
Исходные данные представлены в формате событий: одному клиенту (`id`) соответствует несколько наблюдений, упорядоченных по индексу `rn`.
Цель данного блока — преобразовать событийную таблицу в клиентскую, где каждой строке соответствует один клиент.

## Подход
Обработка выполняется батчами (по parquet-файлам) для контроля использования памяти.
Для каждого клиента рассчитываются следующие группы признаков:

1. **Структура истории:**
   - `loans_cnt` — количество наблюдений (длина истории);
   - `rn_min`, `rn_max` — диапазон временного индекса.

2. **Числовые признаки (кроме enc_):**
   - `sum`, `count`, `min`, `max`;
   - `mean` рассчитывается после финальной агрегации как `sum / count`,
     что обеспечивает корректное объединение между батчами.

3. **Категориальные loan-признаки:**
   - рассчитывается количество наблюдений (`count`);
   - `nunique` не используется в baseline-версии, поскольку его нельзя
     корректно агрегировать между файлами простой операцией.

4. **Платёжные признаки:**
   - `paym_obs` — количество наблюдаемых платёжных статусов;
   - `paym_bad` — количество статусов, отличных от 0 (интерпретируются как отклонения от нормы);
   - `paym_bad_share` — доля проблемных статусов;
   - `paym_obs_is0` — индикатор отсутствия платёжной истории.

5. **Текущее состояние клиента:**
   - `last__enc_*` — значения признаков в последнем наблюдении
     (определяется глобально по максимальному `rn`).

## Результат

Получен агрегированный датасет уровня клиента, объединяющий:
- характеристики истории,
- статистики числовых признаков,
- платёжную дисциплину,
- текущее состояние клиента.

Данный датасет используется как вход для построения модели кредитного риска.


### Сохранение датасета

In [10]:
dataset.to_parquet("final_features.parquet", index=False)

### Загрузка подготовленного датасета без повторного чтения исходных данных

In [6]:
dataset = pd.read_parquet("final_features.parquet")

## Проверка структуры

In [7]:
print("Shape:", dataset.shape)
print("\nDtypes:")
print(dataset.dtypes.value_counts())

print("\nПропуски (top 20):")
display(dataset.isna().mean().sort_values(ascending=False).head(20))

print("\nКоличество константных колонок:")
const_cols = [c for c in dataset.columns if dataset[c].nunique(dropna=False) <= 1]
print(len(const_cols))

Shape: (3000000, 181)

Dtypes:
float64    177
int64        4
Name: count, dtype: int64

Пропуски (top 20):


id                  0.0
enc_paym_8_max      0.0
enc_paym_9_min      0.0
enc_paym_9_max      0.0
enc_paym_10_mean    0.0
enc_paym_10_min     0.0
enc_paym_10_max     0.0
enc_paym_11_mean    0.0
enc_paym_11_min     0.0
enc_paym_11_max     0.0
enc_paym_12_mean    0.0
enc_paym_12_min     0.0
enc_paym_12_max     0.0
enc_paym_13_mean    0.0
enc_paym_13_min     0.0
enc_paym_13_max     0.0
enc_paym_14_mean    0.0
enc_paym_14_min     0.0
enc_paym_14_max     0.0
enc_paym_15_mean    0.0
dtype: float64


Количество константных колонок:
1


## Проверка итогового агрегированного датасета
- Размерность датасета: ~3 млн клиентов и 191 признак.
- Все признаки имеют числовой тип, категориальные признаки представлены в закодированном виде.
- Пропуски в данных отсутствуют, что подтверждает корректность агрегации и обработки делений.
- Обнаружено 2 константных признака, которые будут удалены перед обучением модели.

Таким образом, итоговый датасет является корректным и готовым к построению baseline-модели.

In [8]:
const_cols = [c for c in dataset.columns if dataset[c].nunique(dropna=False) <= 1]
dataset = dataset.drop(columns=const_cols)

### Объединение с таргетом

In [9]:
target = pd.read_csv("train_target.csv")
dataset = final_features.merge(target, on="id", how="inner")
dataset.shape


(3000000, 182)

In [10]:
dataset.head()

,id,loans_cnt,rn_max,rn_min,pre_since_opened_mean,pre_since_opened_min,pre_since_opened_max,pre_since_confirmed_mean,pre_since_confirmed_min,pre_since_confirmed_max,...,enc_loans_account_cur_mean,enc_loans_account_cur_min,enc_loans_account_cur_max,pclose_flag_mean,pclose_flag_min,pclose_flag_max,fclose_flag_mean,fclose_flag_min,fclose_flag_max,flag
0,0,10,10,1,8.100000,1.0,18.0,7.600000,0.0,12.0,...,1.0,1.0,1.0,0.100000,0.0,1.0,0.200000,0.0,1.0,0
1,1,14,14,1,11.428571,2.0,18.0,7.642857,3.0,14.0,...,1.0,1.0,1.0,0.071429,0.0,1.0,0.142857,0.0,1.0,0
2,2,3,3,1,8.333333,0.0,13.0,10.666667,9.0,14.0,...,1.0,1.0,1.0,0.666667,0.0,1.0,0.666667,0.0,1.0,0
3,3,15,15,1,7.000000,1.0,18.0,7.333333,0.0,16.0,...,1.0,1.0,1.0,0.333333,0.0,1.0,0.400000,0.0,1.0,0
4,4,1,1,1,12.000000,12.0,12.0,9.000000,9.0,9.0,...,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,1.0,0


### Проверка баланса классов

In [11]:
print(dataset["flag"].value_counts(normalize=True))

flag
0    0.964519
1    0.035481
Name: proportion, dtype: float64


### Разделение на признаки и target

In [14]:
X = dataset.drop(columns=["id", "flag"])
y = dataset["flag"]

### Train/Test split

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (2400000, 180) Test: (600000, 180)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

model = LogisticRegression(max_iter=1000, class_weight="balanced")

model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, proba)

print("ROC-AUC:", roc)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced"
    ))
])

pipe.fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, proba)
